# IFC Engine Test Analysis Notebook

This notebook replaces the CLI-only flow with an interactive analysis workflow for `engine_test_log_*.csv`.

It provides:
- metadata + quality checks
- spool step metrics (`t63`, `t90`, `t95`)
- steady-state map fits
- model-vs-data comparisons
- plots for all key channels

> If `matplotlib` is not installed in your Python environment, install it and rerun:
> `pip install matplotlib`


In [ ]:
from pathlib import Path
import csv
import math
import json
from dataclasses import dataclass

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception as ex:
    HAS_MPL = False
    MPL_IMPORT_ERROR = ex

print('matplotlib available:', HAS_MPL)
if not HAS_MPL:
    print('matplotlib import error:', MPL_IMPORT_ERROR)


In [ ]:
# --- Configure input log here ---
# Option 1: hardcode a specific file
LOG_PATH = Path(r"../../engine test logs/engine_test_log_00000003.csv")

# Option 2: set LOG_PATH=None to auto-pick latest
AUTO_PICK_LATEST = False

if AUTO_PICK_LATEST or LOG_PATH is None:
    log_dir = Path(r"../../engine test logs")
    candidates = sorted(log_dir.glob("engine_test_log_*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No engine_test_log_*.csv in {log_dir.resolve()}")
    LOG_PATH = candidates[-1]

LOG_PATH = LOG_PATH.resolve()
print('Using log:', LOG_PATH)


In [ ]:
def parse_engine_log(path: Path):
    metadata = {}
    data_lines = []

    with path.open('r', encoding='utf-8', newline='') as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            if s.startswith('#'):
                payload = s[1:].strip()
                if '=' in payload:
                    k, v = payload.split('=', 1)
                    metadata[k.strip()] = v.strip()
                continue
            data_lines.append(line)

    if not data_lines:
        raise ValueError(f'No CSV content in {path}')

    reader = csv.DictReader(data_lines)
    rows = list(reader)
    if not rows:
        raise ValueError('No rows parsed from CSV')

    df = pd.DataFrame(rows)

    # numeric conversion where possible
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='ignore')

    # force key numerics
    for c in ['t_s', 'phase_idx', 'cmd_throttle', 'engine_thrust_kn', 'eng_mod_fuel_flow', 'intake_air_amt']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    df = df.dropna(subset=['t_s', 'cmd_throttle', 'engine_thrust_kn']).copy()
    df = df.sort_values('t_s').reset_index(drop=True)

    return metadata, df

metadata, df = parse_engine_log(LOG_PATH)
print('Rows:', len(df))
print('Columns:', len(df.columns))
metadata


In [ ]:
# Quick quality summary
summary_cols = [
    'cmd_throttle',
    'act_throttle',
    'engine_thrust_kn',
    'engine_avail_thrust_kn',
    'eng_mod_fuel_flow',
    'intake_air_amt',
    'ship_groundspeed_mps',
    'ship_airspeed_mps',
    'ship_alt_m'
]

q = {}
for c in summary_cols:
    if c in df.columns:
        s = pd.to_numeric(df[c], errors='coerce')
        q[c] = {
            'min': float(np.nanmin(s.values)),
            'max': float(np.nanmax(s.values)),
            'mean': float(np.nanmean(s.values))
        }

q


In [ ]:
@dataclass
class Segment:
    start_idx: int
    end_idx: int
    cmd: float

@dataclass
class StepMetric:
    step_index: int
    direction: str
    cmd_from: float
    cmd_to: float
    t_start_s: float
    y0_kn: float
    yss_kn: float
    delta_kn: float
    t63_s: float | None
    t90_s: float | None
    t95_s: float | None

def tail_window(arr, frac=0.30, minimum=5):
    arr = np.asarray(arr, dtype=float)
    n = len(arr)
    if n <= 0:
        return arr
    k = max(minimum, int(round(n * frac)))
    if k >= n:
        return arr
    return arr[-k:]

def build_segments(df_in: pd.DataFrame, eps=1e-4):
    segs = []
    cmd = float(df_in.loc[0, 'cmd_throttle'])
    start = 0
    for i in range(1, len(df_in)):
        nxt = float(df_in.loc[i, 'cmd_throttle'])
        if abs(nxt - cmd) > eps:
            segs.append(Segment(start, i - 1, cmd))
            start = i
            cmd = nxt
    segs.append(Segment(start, len(df_in) - 1, cmd))
    return segs

def find_threshold_time(df_in, start_idx, end_idx, y0, yss, frac):
    delta = yss - y0
    if abs(delta) < 1e-9:
        return None
    target = y0 + frac * delta
    sign = 1.0 if delta > 0 else -1.0
    t0 = float(df_in.loc[start_idx, 't_s'])

    for i in range(start_idx, end_idx + 1):
        y = float(df_in.loc[i, 'engine_thrust_kn'])
        if (y - target) * sign >= 0:
            return float(df_in.loc[i, 't_s']) - t0
    return None

def compute_step_metrics(df_in, segments):
    out = []
    for i in range(1, len(segments)):
        prv = segments[i - 1]
        cur = segments[i]

        y_prev = df_in.loc[prv.start_idx:prv.end_idx, 'engine_thrust_kn'].astype(float).values
        y_cur = df_in.loc[cur.start_idx:cur.end_idx, 'engine_thrust_kn'].astype(float).values
        if len(y_prev) == 0 or len(y_cur) == 0:
            continue

        y0 = float(np.mean(tail_window(y_prev)))
        yss = float(np.mean(tail_window(y_cur)))
        delta = yss - y0
        if abs(delta) < 1e-3:
            continue

        out.append(StepMetric(
            step_index=i,
            direction='up' if delta > 0 else 'down',
            cmd_from=float(prv.cmd),
            cmd_to=float(cur.cmd),
            t_start_s=float(df_in.loc[cur.start_idx, 't_s']),
            y0_kn=y0,
            yss_kn=yss,
            delta_kn=delta,
            t63_s=find_threshold_time(df_in, cur.start_idx, cur.end_idx, y0, yss, 0.632),
            t90_s=find_threshold_time(df_in, cur.start_idx, cur.end_idx, y0, yss, 0.900),
            t95_s=find_threshold_time(df_in, cur.start_idx, cur.end_idx, y0, yss, 0.950),
        ))
    return out

segments = build_segments(df)
steps = compute_step_metrics(df, segments)

pd.DataFrame([s.__dict__ for s in steps])


In [ ]:
def fit_poly_map(points_cmd, points_y, degree=2):
    x = np.asarray(points_cmd, dtype=float)
    y = np.asarray(points_y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2:
        return None

    deg = degree if len(np.unique(np.round(x, 6))) >= 3 else 1
    coefs = np.polyfit(x, y, deg=deg)
    yhat = np.polyval(coefs, x)
    rmse = float(np.sqrt(np.mean((y - yhat) ** 2)))
    return {
        'degree': deg,
        'coefs': coefs,
        'rmse': rmse,
        'x': x,
        'y': y,
        'yhat': yhat,
    }

def segment_steady_points(df_in, segs):
    pts = []
    for seg in segs:
        y = df_in.loc[seg.start_idx:seg.end_idx, 'engine_thrust_kn'].astype(float).values
        mdot = pd.to_numeric(df_in.loc[seg.start_idx:seg.end_idx, 'engine_massflow_kgps'], errors='coerce').values if 'engine_massflow_kgps' in df_in.columns else np.full(len(y), np.nan)
        pts.append({
            'cmd': float(seg.cmd),
            'thrust_kn': float(np.mean(tail_window(y))),
            'mdot_kgps': float(np.nanmean(tail_window(mdot))) if np.any(np.isfinite(mdot)) else np.nan,
        })
    return pd.DataFrame(pts)

steady_df = segment_steady_points(df, segments)
thrust_fit = fit_poly_map(steady_df['cmd'].values, steady_df['thrust_kn'].values, degree=2)
mdot_fit = fit_poly_map(steady_df['cmd'].values, steady_df['mdot_kgps'].values, degree=2)

steady_df, thrust_fit['rmse'] if thrust_fit else None


In [ ]:
# Steady-state reach check per phase
# Tune these thresholds to match your engine/test stand behavior.
STEADY_WINDOW_S = 4.0
STEADY_MIN_POINTS = 12
STEADY_THRUST_SLOPE_MAX = 0.08      # kN/s
STEADY_THRUST_STD_MAX = 0.35        # kN
STEADY_FUEL_SLOPE_MAX = 0.0020      # (eng_mod_fuel_flow units)/s
ENFORCE_STEADY_STATE = False        # set True to hard-fail notebook when any phase is non-steady

rows = []
for seg in segments:
    seg_df = df.loc[seg.start_idx:seg.end_idx].copy()
    if seg_df.empty:
        continue

    t0 = float(seg_df['t_s'].iloc[0])
    t1 = float(seg_df['t_s'].iloc[-1])
    duration_s = t1 - t0

    tail_start = t1 - STEADY_WINDOW_S
    tail = seg_df[seg_df['t_s'] >= tail_start].copy()
    if len(tail) < STEADY_MIN_POINTS:
        tail = seg_df.tail(min(STEADY_MIN_POINTS, len(seg_df))).copy()

    t_tail = tail['t_s'].astype(float).values
    y_tail = tail['engine_thrust_kn'].astype(float).values

    thrust_slope = np.nan
    thrust_std = np.nan
    if len(t_tail) >= 2 and np.all(np.isfinite(y_tail)):
        thrust_slope = float(np.polyfit(t_tail - t_tail[0], y_tail, deg=1)[0])
        thrust_std = float(np.std(y_tail))

    fuel_slope = np.nan
    if 'eng_mod_fuel_flow' in tail.columns:
        f_tail = pd.to_numeric(tail['eng_mod_fuel_flow'], errors='coerce').values
        if len(t_tail) >= 2 and np.sum(np.isfinite(f_tail)) >= 2:
            mask = np.isfinite(f_tail)
            tt = t_tail[mask]
            ff = f_tail[mask]
            fuel_slope = float(np.polyfit(tt - tt[0], ff, deg=1)[0])

    reasons = []
    if not np.isfinite(thrust_slope):
        reasons.append('missing thrust slope')
    elif abs(thrust_slope) > STEADY_THRUST_SLOPE_MAX:
        reasons.append(f'thrust slope {thrust_slope:.3f} kN/s > {STEADY_THRUST_SLOPE_MAX:.3f}')

    if not np.isfinite(thrust_std):
        reasons.append('missing thrust std')
    elif thrust_std > STEADY_THRUST_STD_MAX:
        reasons.append(f'thrust std {thrust_std:.3f} kN > {STEADY_THRUST_STD_MAX:.3f}')

    if np.isfinite(fuel_slope) and abs(fuel_slope) > STEADY_FUEL_SLOPE_MAX:
        reasons.append(f'fuel slope {fuel_slope:.5f} > {STEADY_FUEL_SLOPE_MAX:.5f}')

    reached = len(reasons) == 0

    rows.append({
        'phase_idx': int(round(seg_df['phase_idx'].iloc[0])) if 'phase_idx' in seg_df.columns else -1,
        'phase_name': str(seg_df['phase_name'].iloc[0]) if 'phase_name' in seg_df.columns else 'unknown',
        'cmd': float(seg.cmd),
        'duration_s': float(duration_s),
        'tail_samples': int(len(tail)),
        'thrust_tail_slope_knps': float(thrust_slope) if np.isfinite(thrust_slope) else np.nan,
        'thrust_tail_std_kn': float(thrust_std) if np.isfinite(thrust_std) else np.nan,
        'fuel_tail_slope': float(fuel_slope) if np.isfinite(fuel_slope) else np.nan,
        'reached_steady_state': bool(reached),
        'reasons': '; '.join(reasons) if reasons else 'PASS',
    })

steady_check_df = pd.DataFrame(rows).sort_values('phase_idx').reset_index(drop=True)
all_steady = bool(steady_check_df['reached_steady_state'].all()) if not steady_check_df.empty else False

print('All phases steady:', all_steady)
if not all_steady:
    print('Non-steady phases:')
    print(steady_check_df.loc[~steady_check_df['reached_steady_state'], ['phase_idx', 'phase_name', 'reasons']])

if ENFORCE_STEADY_STATE and not all_steady:
    raise AssertionError('One or more phases did not reach steady state')

steady_check_df


In [ ]:
def median_tau(step_metrics, direction):
    vals = [getattr(s, 't63_s') for s in step_metrics if s.direction == direction and getattr(s, 't63_s') is not None]
    if not vals:
        return np.nan
    return float(np.median(vals))

tau_up = median_tau(steps, 'up')
tau_down = median_tau(steps, 'down')

print('tau_up_t63:', tau_up)
print('tau_down_t63:', tau_down)

# model simulation on full run
if thrust_fit is None:
    raise RuntimeError('No thrust fit available')

def thrust_ss(u):
    return float(np.polyval(thrust_fit['coefs'], u))

t = df['t_s'].astype(float).values
u = df['cmd_throttle'].astype(float).values
y = df['engine_thrust_kn'].astype(float).values

y_model = np.zeros_like(y)
y_model[0] = y[0]

for i in range(1, len(y)):
    dt = max(1e-3, t[i] - t[i - 1])
    yss = thrust_ss(u[i])
    tau = tau_up if yss >= y_model[i - 1] else tau_down
    if not np.isfinite(tau) or tau <= 1e-6:
        tau = 1.0
    y_model[i] = y_model[i - 1] + dt * (yss - y_model[i - 1]) / tau

rmse_model = float(np.sqrt(np.mean((y - y_model) ** 2)))
print('full-run model RMSE (kN):', rmse_model)


In [ ]:
# Plots: key channels + model comparisons
if not HAS_MPL:
    raise RuntimeError('matplotlib is required for plotting in this notebook')

plt.style.use('default')
fig, ax = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

ax[0].plot(df['t_s'], df['cmd_throttle'], label='cmd_throttle', color='tab:blue')
if 'act_throttle' in df.columns:
    ax[0].plot(df['t_s'], df['act_throttle'], label='act_throttle', color='tab:orange', alpha=0.8)
ax[0].set_ylabel('Throttle')
ax[0].grid(True, alpha=0.3)
ax[0].legend(loc='best')

ax[1].plot(df['t_s'], df['engine_thrust_kn'], label='engine_thrust_kn', color='tab:red')
if 'engine_avail_thrust_kn' in df.columns:
    ax[1].plot(df['t_s'], df['engine_avail_thrust_kn'], label='engine_avail_thrust_kn', color='tab:purple', alpha=0.6)
ax[1].set_ylabel('Thrust (kN)')
ax[1].grid(True, alpha=0.3)
ax[1].legend(loc='best')

if 'eng_mod_fuel_flow' in df.columns:
    ax[2].plot(df['t_s'], df['eng_mod_fuel_flow'], label='eng_mod_fuel_flow', color='tab:green')
if 'engine_massflow_kgps' in df.columns:
    ax[2].plot(df['t_s'], df['engine_massflow_kgps'], label='engine_massflow_kgps', color='tab:brown', alpha=0.8)
ax[2].set_ylabel('Flow')
ax[2].grid(True, alpha=0.3)
ax[2].legend(loc='best')

if 'intake_air_amt' in df.columns:
    ax[3].plot(df['t_s'], df['intake_air_amt'], label='intake_air_amt', color='tab:cyan')
if 'ship_airspeed_mps' in df.columns:
    ax[3].plot(df['t_s'], df['ship_airspeed_mps'], label='ship_airspeed_mps', color='tab:gray', alpha=0.8)
ax[3].set_ylabel('Intake / Airspeed')
ax[3].set_xlabel('Time (s)')
ax[3].grid(True, alpha=0.3)
ax[3].legend(loc='best')

for s in steps:
    for a in ax:
        a.axvline(s.t_start_s, color='k', alpha=0.08)

fig.suptitle('Engine Test Channels')
fig.tight_layout()
plt.show()


In [ ]:
# Model vs measured thrust over full run
if not HAS_MPL:
    raise RuntimeError('matplotlib is required for plotting in this notebook')

fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax[0].plot(t, y, label='Measured thrust', color='tab:red')
ax[0].plot(t, y_model, label='Model thrust', color='tab:blue', alpha=0.9)
ax[0].set_ylabel('Thrust (kN)')
ax[0].set_title(f'Model vs Data (RMSE={rmse_model:.3f} kN)')
ax[0].grid(True, alpha=0.3)
ax[0].legend(loc='best')

resid = y - y_model
ax[1].plot(t, resid, color='tab:orange', label='Residual')
ax[1].axhline(0.0, color='k', lw=1)
ax[1].set_ylabel('Error (kN)')
ax[1].set_xlabel('Time (s)')
ax[1].grid(True, alpha=0.3)
ax[1].legend(loc='best')

for s in steps:
    ax[0].axvline(s.t_start_s, color='k', alpha=0.08)
    ax[1].axvline(s.t_start_s, color='k', alpha=0.08)

fig.tight_layout()
plt.show()


In [ ]:
# Steady-state map fit plot (throttle -> thrust)
if not HAS_MPL:
    raise RuntimeError('matplotlib is required for plotting in this notebook')

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(steady_df['cmd'], steady_df['thrust_kn'], s=70, label='Segment steady points', color='tab:red')

xg = np.linspace(0, 1, 200)
yg = np.polyval(thrust_fit['coefs'], xg)
ax.plot(xg, yg, label=f"Fit deg={thrust_fit['degree']} (RMSE={thrust_fit['rmse']:.3f} kN)", color='tab:blue')

ax.set_xlabel('Throttle command')
ax.set_ylabel('Steady thrust (kN)')
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
ax.set_title('Steady-State Thrust Map')
plt.show()


In [ ]:
# Step-response overlays (normalized) + first-order reference by direction
if not HAS_MPL:
    raise RuntimeError('matplotlib is required for plotting in this notebook')

def extract_norm_step(df_in, seg_prev, seg_cur):
    y_prev = df_in.loc[seg_prev.start_idx:seg_prev.end_idx, 'engine_thrust_kn'].astype(float).values
    y_cur = df_in.loc[seg_cur.start_idx:seg_cur.end_idx, 'engine_thrust_kn'].astype(float).values

    y0 = float(np.mean(tail_window(y_prev)))
    yss = float(np.mean(tail_window(y_cur)))
    delta = yss - y0
    if abs(delta) < 1e-6:
        return None

    tseg = df_in.loc[seg_cur.start_idx:seg_cur.end_idx, 't_s'].astype(float).values
    yseg = df_in.loc[seg_cur.start_idx:seg_cur.end_idx, 'engine_thrust_kn'].astype(float).values
    tn = tseg - tseg[0]
    yn = (yseg - y0) / delta

    direction = 'up' if delta > 0 else 'down'
    return direction, tn, yn

norm_steps = []
for i in range(1, len(segments)):
    ns = extract_norm_step(df, segments[i-1], segments[i])
    if ns is not None:
        norm_steps.append(ns)

fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for direction, tn, yn in norm_steps:
    idx = 0 if direction == 'up' else 1
    ax[idx].plot(tn, yn, alpha=0.45)

# Reference first-order curves from median tau
tref = np.linspace(0, 12, 300)
if np.isfinite(tau_up) and tau_up > 0:
    ax[0].plot(tref, 1 - np.exp(-tref / tau_up), color='k', lw=2.2, label=f'1st-order ref tau={tau_up:.2f}s')
if np.isfinite(tau_down) and tau_down > 0:
    ax[1].plot(tref, 1 - np.exp(-tref / tau_down), color='k', lw=2.2, label=f'1st-order ref tau={tau_down:.2f}s')

ax[0].set_title('Up-steps normalized')
ax[1].set_title('Down-steps normalized')
for a in ax:
    a.set_xlabel('t from step (s)')
    a.set_ylabel('Normalized response')
    a.grid(True, alpha=0.3)
    a.set_ylim(-0.2, 1.2)
    a.legend(loc='best')

plt.tight_layout()
plt.show()


In [ ]:
# Optional: export model JSON next to log
model = {
    'source_log': str(LOG_PATH),
    'metadata': metadata,
    'tau_up_t63_s': float(tau_up),
    'tau_down_t63_s': float(tau_down),
    'thrust_fit_degree': int(thrust_fit['degree']) if thrust_fit else None,
    'thrust_fit_coefs': [float(x) for x in thrust_fit['coefs']] if thrust_fit else None,
    'thrust_fit_rmse_kn': float(thrust_fit['rmse']) if thrust_fit else None,
    'model_rmse_kn': float(rmse_model),
    'step_metrics': [s.__dict__ for s in steps],
    'steady_state_check': steady_check_df.to_dict(orient='records') if 'steady_check_df' in globals() else None,
    'all_phases_steady': bool(all_steady) if 'all_steady' in globals() else None,
}

out_model = LOG_PATH.with_name(LOG_PATH.stem + '_model_from_notebook.json')
out_model.write_text(json.dumps(model, indent=2), encoding='utf-8')
out_model
